# Lab 06: A diffusion model from scratch on 2-D toy data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hokwangchoi/hc-model-study/blob/master/labs/notebooks/lab06_diffusion.ipynb) · lab page: [`labs/lab06.html`](../lab06.html) · file: [`labs/lab06_diffusion.py`](../lab06_diffusion.py)

**Time:** 60-90 min

The last step trains a model for about 5-20 s on a CPU.

## Goal

Implement the pieces every diffusion model shares (DDPM training, DDIM
sampling) and train a tiny MLP denoiser on a 2-D distribution: 8 Gaussian
blobs on a ring. If everything is right, pure noise turns into points on
the 8 blobs. The same maths drives DiT and Stable Diffusion; only the
network (a transformer on image latents) is bigger.

## The maths, step by step

```text
T = 1000 noise levels, t = 0 .. T-1 (t = 0 is almost clean).

  1. Noise schedule (DDPM / DiT linear schedule)
       beta_t     = linspace(1e-4, 0.02, T)[t]          noise added at step t
       alpha_t    = 1 - beta_t
       abar_t     = alpha_0 * alpha_1 * ... * alpha_t   (cumulative product)
     abar_t falls from ~1 (clean) to ~4e-5 (pure noise).
  2. Forward noising in closed form, q(x_t | x_0):
       x_t = sqrt(abar_t) * x_0 + sqrt(1 - abar_t) * eps,   eps ~ N(0, I)
     No loop over t: any noise level is one line away.
  3. Training objective (epsilon prediction):
       sample x_0 from the data, t ~ Uniform{0..T-1}, eps ~ N(0, I)
       loss = || eps_theta(x_t, t) - eps ||^2   (mean squared error)
     The network learns to name the noise that was added.
  4. DDIM sampling (deterministic, sigma = 0), on a subsequence of steps,
     e.g. 50 steps: t = 999, 979, ..., 19 and then "t = -1" with abar = 1:
       eps_hat = eps_theta(x_t, t)
       x0_hat  = (x_t - sqrt(1 - abar_t) * eps_hat) / sqrt(abar_t)
       x_next  = sqrt(abar_next) * x0_hat + sqrt(1 - abar_next) * eps_hat
     Start from x_T ~ N(0, I); the last step (abar_next = 1) returns x0_hat.

  Worked example from the guide (one pixel): x_0 = 0.8, eps = -1.2, t = 500:
       sqrt(abar) = 0.2789, sqrt(1 - abar) = 0.9603, x_500 = -0.9292,
       and one DDIM step to t = 250 with the true eps gives x_250 = -0.2525.
```

## Read alongside

- [Diffusion guide](../../diffusion/diffusion_guide.html): "The Core Idea: Iterative Denoising", 1. Forward Process (1.1 the noise schedule, #noise-widget), 3.1 DDIM, 5. Code Walkthrough

## How to work

1. Run the setup cell and the imports cell.
2. Implement the functions in order, replacing `raise NotImplementedError  # TODO`. The hints above each cell open one at a time; try Hint 1 first.
3. Run each cell after editing it, then run the check cell at the end: `check.run_lab(6, namespace=globals())` checks the functions defined in this notebook (TODO = not started, PASS = matches the reference, FAIL = read the message).

Your work stays in this notebook; it does not change `labs/lab06_diffusion.py` or the lab page. The reference solution is `labs/solutions/lab06_diffusion.py`.

In [ ]:
# Setup: make the checker (labs/check.py) importable. Run this cell first.
# In a clone of the repository (Jupyter, VS Code) it uses labs/check.py;
# elsewhere (e.g. Google Colab) it downloads check.py, which needs only torch.
import pathlib
import sys
import urllib.request


def _find_labs_dir():
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        for cand in (d, d / "labs"):
            if (cand / "check.py").is_file() and (cand / "solutions").is_dir():
                return cand
    return None


LABS_DIR = _find_labs_dir()
if LABS_DIR is None:
    LABS_DIR = pathlib.Path("hc_model_study_labs").resolve()
    LABS_DIR.mkdir(exist_ok=True)
    if not (LABS_DIR / "check.py").is_file():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/hokwangchoi/hc-model-study/master/labs/check.py", LABS_DIR / "check.py")
sys.path.insert(0, str(LABS_DIR))
import check  # noqa: E402

import torch  # noqa: E402
print("checker:", LABS_DIR / "check.py", "| torch", torch.__version__)

### Imports

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

## Step 1: the noise schedule

**Hints for `linear_beta_schedule`** (open one at a time):

<details><summary>Hint 1</summary>

```text
torch.linspace includes both end points.
```

</details>

In [ ]:
def linear_beta_schedule(T: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    """betas [T] (float32), linearly spaced from beta_start to beta_end inclusive."""
    raise NotImplementedError  # TODO

**Hints for `alpha_bars_from_betas`** (open one at a time):

<details><summary>Hint 1</summary>

```text
torch.cumprod(..., dim=0)
```

</details>

In [ ]:
def alpha_bars_from_betas(betas: torch.Tensor) -> torch.Tensor:
    """abar [T]: abar_t = prod_{s <= t} (1 - beta_s)."""
    raise NotImplementedError  # TODO

## Step 2: forward noising q(x_t | x_0)

**Hints for `q_sample`** (open one at a time):

<details><summary>Hint 1</summary>

```text
alpha_bars[t] is [B]; it must broadcast against [B, ...].
```

</details>

<details><summary>Hint 2</summary>

```text
ab = alpha_bars[t].view(-1, *([1] * (x0.dim() - 1)))   -> [B, 1, ...]
```

</details>

<details><summary>Hint 3</summary>

```text
return ab.sqrt() * x0 + (1 - ab).sqrt() * noise
```

</details>

In [ ]:
def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor,
             alpha_bars: torch.Tensor) -> torch.Tensor:
    """x_t = sqrt(abar_t) x0 + sqrt(1 - abar_t) noise.

    x0, noise: [B, ...] (any trailing shape: [B, 2] points or [B, C, H, W] images)
    t:         [B] int64, one noise level per sample
    """
    raise NotImplementedError  # TODO

## Step 3: the epsilon-prediction loss

**Hints for `epsilon_loss`** (open one at a time):

<details><summary>Hint 1</summary>

```text
build x_t with your q_sample, run the model, compare with the noise.
```

</details>

<details><summary>Hint 2</summary>

```text
return F.mse_loss(model(q_sample(x0, t, noise, alpha_bars), t), noise)
```

</details>

In [ ]:
def epsilon_loss(model: nn.Module, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor,
                 alpha_bars: torch.Tensor) -> torch.Tensor:
    """Scalar MSE between model(x_t, t) and the noise that produced x_t."""
    raise NotImplementedError  # TODO

## Given: the denoiser network and the toy data

Given: `timestep_embedding(t, dim, max_period=10000.0)` is already written. Run the cell.

In [ ]:
def timestep_embedding(t: torch.Tensor, dim: int, max_period: float = 10000.0) -> torch.Tensor:
    """Sinusoidal embedding of integer timesteps: [B] -> [B, dim] (as in DiT)."""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

Given: `class Denoiser(nn.Module)` is already written. Run the cell.

In [ ]:
class Denoiser(nn.Module):
    """eps_theta(x_t, t): a 3-hidden-layer MLP on [x_t, embed(t)]. Tiny stand-in for a DiT."""

    def __init__(self, dim: int = 2, hidden: int = 128, t_dim: int = 32):
        super().__init__()
        self.t_dim = t_dim
        self.layers = nn.ModuleList([nn.Linear(dim + t_dim, hidden), nn.Linear(hidden, hidden),
                                     nn.Linear(hidden, hidden), nn.Linear(hidden, dim)])

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """x_t [B, dim], t [B] int64 -> predicted noise [B, dim]"""
        h = torch.cat([x_t, timestep_embedding(t, self.t_dim)], dim=-1)
        for layer in self.layers[:-1]:
            h = F.silu(layer(h))
        return self.layers[-1](h)

Given: `make_ring_data(n, generator=None, n_modes=8, radius=2.0, std=0.1)` is already written. Run the cell.

In [ ]:
def make_ring_data(n: int, generator: torch.Generator = None, n_modes: int = 8,
                   radius: float = 2.0, std: float = 0.1) -> torch.Tensor:
    """n points [n, 2]: a mixture of n_modes Gaussians evenly spaced on a circle."""
    k = torch.randint(0, n_modes, (n,), generator=generator)
    angle = 2 * math.pi * k / n_modes
    centres = torch.stack([radius * torch.cos(angle), radius * torch.sin(angle)], dim=-1)
    return centres + std * torch.randn(n, 2, generator=generator)

## Step 4: the training loop

**Hints for `train_denoiser`** (open one at a time):

<details><summary>Hint 1</summary>

```text
g = torch.Generator().manual_seed(seed); opt = torch.optim.Adam(model.parameters(), lr=lr)
```

</details>

<details><summary>Hint 2</summary>

```text
per step: idx = torch.randint(0, len(data), (batch_size,), generator=g)
t = torch.randint(0, len(alpha_bars), (batch_size,), generator=g)
noise = torch.randn(batch_size, data.shape[1], generator=g)
```

</details>

<details><summary>Hint 3</summary>

```text
loss = epsilon_loss(model, data[idx], t, noise, alpha_bars)
opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
```

</details>

In [ ]:
def train_denoiser(model: nn.Module, data: torch.Tensor, alpha_bars: torch.Tensor,
                   steps: int = 3000, batch_size: int = 512, lr: float = 2e-3,
                   seed: int = 0) -> list:
    """Train `model` with epsilon_loss on random minibatches of `data` [N, 2].

    Each step: pick batch_size random rows of data, t ~ Uniform{0..T-1},
    noise ~ N(0, I), loss = epsilon_loss(...), Adam update.
    Use a torch.Generator seeded with `seed` for every random draw.
    Returns the list of per-step losses (Python floats).
    """
    raise NotImplementedError  # TODO

## Step 5: DDIM sampling

**Hints for `ddim_timesteps`** (open one at a time):

<details><summary>Hint 1</summary>

```text
stride = T // num_steps; the values are k * stride - 1 for k = num_steps .. 1.
```

</details>

<details><summary>Hint 2</summary>

```text
torch.arange(num_steps, 0, -1) * stride - 1
```

</details>

In [ ]:
def ddim_timesteps(T: int, num_steps: int) -> torch.Tensor:
    """Descending int64 timesteps with stride T // num_steps, ending at stride - 1.

    ddim_timesteps(1000, 50) -> [999, 979, 959, ..., 39, 19]  (50 values)
    ddim_timesteps(1000, 4)  -> [999, 749, 499, 249]
    """
    raise NotImplementedError  # TODO

**Hints for `ddim_step`** (open one at a time):

<details><summary>Hint 1</summary>

```text
two lines: estimate x0, then re-noise it to the next level using the
SAME eps_hat (that is what makes DDIM deterministic).
```

</details>

<details><summary>Hint 2</summary>

```text
abar_t = torch.as_tensor(abar_t) lets you call .sqrt() on floats too.
```

</details>

In [ ]:
def ddim_step(x_t: torch.Tensor, eps_hat: torch.Tensor, abar_t, abar_next) -> torch.Tensor:
    """One deterministic DDIM update from level abar_t to level abar_next.

    abar_t, abar_next: floats or 0-dim tensors. Returns x_next (same shape as x_t).
    """
    raise NotImplementedError  # TODO

**Hints for `ddim_sample`** (open one at a time):

<details><summary>Hint 1</summary>

```text
ts = ddim_timesteps(len(alpha_bars), num_steps); loop with enumerate.
```

</details>

<details><summary>Hint 2</summary>

```text
t_batch = torch.full((x.shape[0],), int(t), dtype=torch.long)
```

</details>

<details><summary>Hint 3</summary>

```text
abar_next = alpha_bars[ts[i + 1]] if i + 1 < len(ts) else 1.0
x = ddim_step(x, model(x, t_batch), alpha_bars[t], abar_next)
```

</details>

In [ ]:
@torch.no_grad()
def ddim_sample(model: nn.Module, x_T: torch.Tensor, alpha_bars: torch.Tensor,
                num_steps: int = 50) -> torch.Tensor:
    """Run DDIM from x_T [n, ...] (pure noise) to x_0.

    Visit t in ddim_timesteps(T, num_steps). At each t call
    model(x, t_batch) with t_batch = an int64 tensor [n] filled with t;
    the next level is the next timestep in the list, and after the last
    one it is abar = 1.0 (clean).
    """
    raise NotImplementedError  # TODO

## Check

Run this after each function you finish. Steps you have not started show TODO; finished steps show PASS. For a full traceback of a failing step: `check.run_lab(6, globals(), verbose=True)`.

In [ ]:
check.run_lab(6, namespace=globals());

## Scratch

Try your functions here: print shapes, call them on small inputs. (Functions you have not written yet raise `NotImplementedError`.)

In [ ]:
# Try the pieces here (Run the cells above first).
betas = linear_beta_schedule()
abar = alpha_bars_from_betas(betas)
print(abar[[0, 250, 500, 999]])
x0 = make_ring_data(4)
q_sample(x0, torch.tensor([0, 250, 500, 999]), torch.randn(4, 2), abar)